In [140]:
# Packages
import os
import re

# For downloading online NOAA data
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor
import requests

# For data analysis and visualization
import pandas as pd
import numpy as np
import zipfile
import matplotlib.pyplot as plt
import geopandas as gpd

# pd.set_option('display.max_colwidth', None)

In [ ]:
# Links of gzip storm data from NOAA website: https://www.ncei.noaa.gov/stormevents/ftp.jsp

def storm_data():
    # Web scrape NOAA weather data links
    storms_url = 'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/'
    req = requests.get(storms_url)                  # access url webpage
    soup = BeautifulSoup(req.text, 'html.parser')   # parse thru HTML text of webpage

    # Find 2000s data.csv.gz filenames in <a href="link" > format 
    pattern = r'StormEvents_details-ftp_v1\.0_d20\d{2}_c\d{8}\.csv\.gz'   # 2000s filename pattern
    data_links = []
    for link in soup.find_all('a', attrs={'href': re.compile(pattern)}):
        year_data = link.get('href')
        full_link = str(storms_url) + str(year_data)
        data_links.append(full_link)
    return data_links


# Parallel download func for data
def download_files(data):
    # Create new directory for data
    wx_data_dir = '../data/noaa'
    os.makedirs(wx_data_dir, exist_ok=True)
    
    # Check url request for 'content-disposition' header to parse .gz filenames
    response = requests.get(data, stream=True)
    if 'content-disposition' in response.headers:
        content_disp = response.headers['content-disposition']
        file_name = content_disp.split('filename=')[1]
    else:
        file_name = data.split('/')[-1]
    
    # Write downloaded gzip data to data dir
    gz_name = os.path.join(wx_data_dir, file_name)
    with open(gz_name, 'wb') as gz_file:
        gz_file.write(response.content)
    # print(f'Downloaded file to {gz_name}')


# Use ThreadPoolExecutor() to parallel download gzip files
with ThreadPoolExecutor() as executor:
    executor.map(download_files, storm_data())

In [ ]:
# Create pandas dataframes of data for each year (optional: state)
# Info about files: https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/Storm-Data-Bulk-csv-Format.pdf 
def weather_df(year, state=None):
    # Look for selected year's data from data/noaa dir
    file_pattern = rf'StormEvents_details-ftp_v1\.0_d{year}_c\d{{8}}\.csv\.gz'
    gz_files = os.listdir('../data/noaa')
    try:
        match = [g for g in gz_files if re.search(file_pattern, g)][0]
    except IndexError:
        print('No matches found! Did you select a year between 2000 and 2026?')
    
    # Adjust date columns to YYYMMDD format
    df = pd.read_csv(f'../data/noaa/{match}', compression='gzip', header=0)
    df['BEGIN_DAY'] = df['BEGIN_DAY'].apply(lambda x: '0'+str(x) if len(str(x))<2 else str(x))
    df['BEGIN_DATE'] = df['BEGIN_YEARMONTH'].astype(str) + df['BEGIN_DAY']
    df['END_DAY'] = df['END_DAY'].apply(lambda x: '0'+str(x) if len(str(x))<2 else str(x))
    df['END_DATE'] = df['END_YEARMONTH'].astype(str) + df['END_DAY']
    
    # Adjust for details & events columns
    ## EVENT_IDs are unique for each entry
    ## MAGNITUDE: wind speeds (knots), hail (inches)
    details = ['EVENT_ID', 'BEGIN_DATE', 'END_DATE', 'STATE', 'STATE_FIPS', 
               'CZ_NAME', 'CZ_FIPS','EVENT_TYPE', 'INJURIES_DIRECT', 
               'INJURIES_INDIRECT', 'DEATHS_DIRECT', 'DEATHS_INDIRECT', 
               'DAMAGE_PROPERTY', 'MAGNITUDE', 'TOR_F_SCALE', 'EPISODE_NARRATIVE']
    event_types = ['Blizzard', 'Cold/Wind Chill', 'Drought', 'Excessive Heat',
                   'Extreme Cold/Wind Chill', 'Flash Flood', 'Flood', 'Hail', 'Heat', 
                   'Heavy Rain', 'Heavy Snow', 'Hurricane (Typhoon)', 'Ice Storm',
                   'Sleet', 'Storm Surge/Tide', 'Thunderstorm Wind', 'Tornado',
                   'Tropical Storm', 'Tsunami', 'Wildfire', 'Winter Storm', 'Winter Weather']
    df = df[details]
    df = df[df['EVENT_TYPE'].isin(event_types)]
    
    # Make rows for each date that events continued through 
    df['days'] = df['END_DATE'].astype(int) - df['BEGIN_DATE'].astype(int)
    rdf = pd.DataFrame(np.repeat(df.values, repeats=df['days']+1, axis=0), columns=df.columns)
    rdf['repeat'] = 1
    rdf['repeat'] = rdf.groupby('EVENT_ID')['repeat'].cumsum() - 1
    rdf['BEGIN_DATE'] = rdf['BEGIN_DATE'].astype(int) + rdf['repeat']
    rdf = rdf.drop(columns=['END_DATE', 'repeat', 'days'])
    rdf = rdf.rename(columns={'BEGIN_DATE': 'DATE'})
    wx_df = rdf
    
    # Filter by state if desired
    if state is not None:
        state = state.upper()
        wx_df = wx_df[wx_df['STATE'] == state]
    
    wx_df = wx_df.sort_values(by=['DATE'], ascending=True).reset_index(drop=True)
    return wx_df


us_2011 = weather_df(2025)
us_2011

In [ ]:
with zipfile.ZipFile('../data/geodata/tl_2025_us_state.zip', 'r') as states:
    states.extractall('../data/geodata')
geo = gpd.read_file('../data/geodata/tl_2025_us_state.shp')
geo.columns
geo['']

Index(['REGION', 'DIVISION', 'STATEFP', 'STATENS', 'GEOID', 'GEOIDFQ',
       'STUSPS', 'NAME', 'LSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER',
       'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='str')